# Multi-decade landfall density

Grid the density of North Atlantic track **fixes** over several decades within a US Gulf/East-coast bounding box to show climatological cyclone exposure. Uses point mode (one feature per fix) and a 2-D histogram.

IBTrACS gives the longest record; the first load is slow. Needs `pip install earthlens[tropycal]`.

## Setup

Consolidate the imports up front. `earthlens` provides the unified `EarthLens` entry point; `matplotlib` renders the density grid. We also create the output directory the tropycal backend writes into.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from earthlens.core import EarthLens

OUT_DIR = Path('tropycal_output')
OUT_DIR.mkdir(exist_ok=True)

## 1 · Load the track fixes

tropycal is a *vector* backend, so we build the request first — basin (`variables`), the IBTrACS source, a 1980-2020 window, and a US Gulf/East-coast bounding box. Keeping the constructor on its own line makes the request easy to read and re-run.

In [ ]:
fixes = None
tracks = EarthLens(
    variables=['north_atlantic'],
    data_source='tropycal',
    start='1980-01-01',
    end='2020-12-31',
    aoi=[-100.0, 20.0, -70.0, 40.0],
    source='ibtracs',
    path=str(OUT_DIR),
)

### Download the fixes

`download()` returns a `GeoDataFrame` with one row per track fix. The first IBTrACS load downloads and parses the whole basin best-track file, so it is slow. We wrap the call in a graceful-skip guard so the notebook still renders when the live query is unavailable.

In [ ]:
try:
    fixes = tracks.download(progress_bar=False)
    print(len(fixes), 'fixes over 1980-2020')
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

## 2 · Grid the fix density

Bin the point fixes into a hexagonal grid with `hexbin`: each cell's colour is the number of track fixes that fall inside it, so denser cells mark the most cyclone-exposed coastline. The plot only runs when the download above succeeded.

In [ ]:
if fixes is not None and len(fixes):
    fig, ax = plt.subplots(figsize=(9, 6))
    hb = ax.hexbin(
        fixes.geometry.x, fixes.geometry.y, gridsize=30, cmap='magma', mincnt=1
    )
    fig.colorbar(hb, ax=ax, label='track fixes per cell')
    ax.set_title('North Atlantic cyclone fix density, 1980-2020')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.show()